In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/features.csv")

train = df[df["season"] <= 2022]
val   = df[df["season"].isin([2023, 2024])]
test  = df[df["season"] == 2025]

print("Train:", len(train), "rows")
print("Validation:", len(val), "rows")
print("Test:", len(test), "rows")

Train: 2922 rows
Validation: 919 rows
Test: 479 rows


In [2]:
def top3_hits(data, score_col):
    """For each race, pick the 3 drivers with the highest score,
    count how many were on the real podium, then average over races."""
    hits = []
    for _, race in data.groupby(["season", "round"]):
        top3 = race.nlargest(3, score_col)
        hits.append(top3["podium"].sum())
    return np.mean(hits)

# Baseline: "the 3 drivers starting at the front finish on the podium"
for name, data in [("Train", train), ("Validation", val), ("Test", test)]:
    data = data.copy()
    data["score"] = -data["grid"]      # lower grid slot = higher score
    print(name, "-> avg podium drivers correct per race (out of 3):",
          round(top3_hits(data, "score"), 2))

Train -> avg podium drivers correct per race (out of 3): 2.1
Validation -> avg podium drivers correct per race (out of 3): 1.96
Test -> avg podium drivers correct per race (out of 3): 2.25


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

features = ["grid", "quali_position", "driver_podium_rate", "driver_avg_finish",
            "driver_dnf_rate", "driver_races", "driver_circuit_podium_rate",
            "constructor_podium_rate", "constructor_circuit_podium_rate"]

X_train, y_train = train[features], train["podium"]

model = make_pipeline(
    SimpleImputer(strategy="mean"),      # fill the blanks with the average
    StandardScaler(),                    # put all features on the same scale
    LogisticRegression(max_iter=1000),
)
model.fit(X_train, y_train)

for name, data in [("Train", train), ("Validation", val), ("Test", test)]:
    data = data.copy()
    data["score"] = model.predict_proba(data[features])[:, 1]
    print(name,
          "| top-3 hits:", round(top3_hits(data, "score"), 2),
          "| ROC-AUC:", round(roc_auc_score(data["podium"], data["score"]), 3))

Train | top-3 hits: 2.07 | ROC-AUC: 0.94
Validation | top-3 hits: 1.65 | ROC-AUC: 0.909
Test | top-3 hits: 1.75 | ROC-AUC: 0.907


In [4]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=6, min_samples_leaf=20,
        random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        subsample=0.8, eval_metric="logloss", random_state=42),
}

imputer = SimpleImputer(strategy="mean")
X_train_imp = imputer.fit_transform(train[features])

for model_name, m in models.items():
    m.fit(X_train_imp, y_train)
    for name, data in [("Validation", val), ("Test", test)]:
        data = data.copy()
        data["score"] = m.predict_proba(imputer.transform(data[features]))[:, 1]
        print(model_name, "|", name,
              "| top-3 hits:", round(top3_hits(data, "score"), 2),
              "| ROC-AUC:", round(roc_auc_score(data["podium"], data["score"]), 3))

Random Forest | Validation | top-3 hits: 1.93 | ROC-AUC: 0.921
Random Forest | Test | top-3 hits: 2.21 | ROC-AUC: 0.935
XGBoost | Validation | top-3 hits: 1.87 | ROC-AUC: 0.919
XGBoost | Test | top-3 hits: 2.29 | ROC-AUC: 0.935


In [5]:
df = pd.read_csv("../data/processed/features.csv")
train = df[df["season"] <= 2022]
val   = df[df["season"].isin([2023, 2024])]
test  = df[df["season"] == 2025]

features2 = features + ["constructor_season_podium_rate", "driver_season_podium_rate",
                        "driver_season_avg_finish", "grid_gap_to_pole", "team_rank_in_race"]

imputer2 = SimpleImputer(strategy="mean")
X_tr = imputer2.fit_transform(train[features2])

models2 = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=6,
                                            min_samples_leaf=20, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                             subsample=0.8, eval_metric="logloss", random_state=42),
}

scaler2 = StandardScaler().fit(X_tr)
for model_name, m in models2.items():
    Xfit = scaler2.transform(X_tr) if model_name == "Logistic Regression" else X_tr
    m.fit(Xfit, train["podium"])
    for name, data in [("Validation", val), ("Test", test)]:
        data = data.copy()
        Xd = imputer2.transform(data[features2])
        if model_name == "Logistic Regression":
            Xd = scaler2.transform(Xd)
        data["score"] = m.predict_proba(Xd)[:, 1]
        print(model_name, "|", name,
              "| top-3 hits:", round(top3_hits(data, "score"), 2),
              "| ROC-AUC:", round(roc_auc_score(data["podium"], data["score"]), 3))

Logistic Regression | Validation | top-3 hits: 1.83 | ROC-AUC: 0.916
Logistic Regression | Test | top-3 hits: 1.92 | ROC-AUC: 0.916
Random Forest | Validation | top-3 hits: 1.96 | ROC-AUC: 0.927
Random Forest | Test | top-3 hits: 2.25 | ROC-AUC: 0.948
XGBoost | Validation | top-3 hits: 1.87 | ROC-AUC: 0.922
XGBoost | Test | top-3 hits: 2.12 | ROC-AUC: 0.942
